# 🌍 TravelDude — Jupyter Notebook
AI-powered travel recommendations with RAG, powered by **Ollama (llama3)** running locally.

**Prerequisites before running:**
```bash
ollama pull llama3   # download the model (~4GB, one time only)
ollama serve         # start the local server
```

## 0 — Setup & Health Check

In [ ]:
import sys, os
sys.path.insert(0, '.')   # add project root to path

# Check Ollama is running
from llm_layer import _ollama_available, OLLAMA_MODEL, OLLAMA_HOST
if _ollama_available():
    print(f'✅ Ollama is running at {OLLAMA_HOST}')
    print(f'   Model: {OLLAMA_MODEL}')
else:
    print('❌ Ollama is NOT running.')
    print('   Start it with: ollama serve')
    print('   Pull model with: ollama pull llama3')

## 1 — Initialise Database

In [ ]:
from database_init import main as init_db
init_db()

## 2 — Seed RAG Index

In [ ]:
from rag.ingest import seed_demo_data, ingest_notes_dir, ingest_reviews_dir
from rag.vector_store import get_index_stats

stats = get_index_stats()
if stats['total_chunks'] == 0:
    print('Seeding demo data...')
    seed_demo_data()
else:
    print(f'RAG index already has {stats["total_chunks"]} chunks')
    print(f'By type: {stats["by_type"]}')

# Optionally ingest your own files:
# ingest_notes_dir('data/notes/')
# ingest_reviews_dir('data/reviews/')

## 3 — Get Recommendations

In [ ]:
from engine import recommend_from_categories

# Edit these preferences (0.0 = not interested, 1.0 = love it)
preferences = {
    'beach':       0.3,
    'food':        0.9,
    'culture':     0.8,
    'adventure':   0.4,
    'nature':      0.5,
    'urban':       0.7,
    'history':     0.6,
}

DB_PATH = 'database/TravelDude.db'
recommendations = recommend_from_categories(
    category_scores=preferences,
    db_path=DB_PATH,
    top_n=5,
)

print('🌍 Top destinations for you:\n')
for i, d in enumerate(recommendations, 1):
    print(f"{i}. {d['name']}, {d['country']} — {d['similarity']*100:.1f}% match | Rating: {d.get('rating','?')} | Cost: {'💰'*d.get('avg_cost_level',1)}")

## 4 — Extract Preferences from Plain English (LLM)

In [ ]:
from llm_layer import extract_preferences_from_text

# Describe your ideal trip in plain English
user_input = "I love exploring local street food markets, learning about history, and wandering through old city neighbourhoods. Not a beach person."

print('Asking llama3 to extract preferences...')
nlp_prefs = extract_preferences_from_text(user_input)
print('\nExtracted preferences:')
for k, v in sorted(nlp_prefs.items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 20)
    print(f'  {k:15s} {bar} {v:.2f}')

## 5 — Why These Destinations? (LLM Narrative)

In [ ]:
from llm_layer import generate_recommendation_narrative

print('Generating narrative with llama3...\n')
narrative = generate_recommendation_narrative(
    user_preferences=preferences,
    destinations=recommendations,
)
print(narrative)

## 6 — RAG-Enriched Itinerary

In [ ]:
from rag.rag_llm import rag_generate_itinerary

# Change destination and days as you like
destination = recommendations[0]['name']
days = 5

print(f'Generating {days}-day RAG-grounded itinerary for {destination}...\n')
itinerary = rag_generate_itinerary(
    destination=destination,
    days=days,
    preferences=preferences,
    pois=[],
)
print(itinerary)

## 7 — RAG Chat (Multi-turn Q&A)

In [ ]:
from rag.rag_llm import TravelChat

chat = TravelChat(destination=destination)
print(f'RAG Chat — asking about {destination}\n')

questions = [
    'What is the best neighbourhood to stay in?',
    'What are the top food experiences I should not miss?',
    'Any tips for getting around cheaply?',
]

for q in questions:
    print(f'Q: {q}')
    answer = chat.ask(q)
    print(f'A: {answer}\n')
    print('-' * 60)

## 8 — Add Your Own Notes to RAG

In [ ]:
from rag.vector_store import index_document, get_index_stats

# Paste your own travel notes directly here
my_note = """
Tokyo tip: The teamLab Borderless museum is unmissable — book 2 weeks ahead.
Best ramen: Fuunji in Shinjuku for tsukemen. Ichiran for late night solo bowls.
Shibuya Sky rooftop > Tokyo Skytree for photos because you get the Skytree in the frame.
Hidden gem: Yanaka neighbourhood for old Tokyo vibes with zero tourists.
"""

n = index_document(
    content=my_note,
    destination='Tokyo',
    source_type='note',
    title='My Tokyo Tips',
)
print(f'Indexed {n} chunks')
print(f'Total index size: {get_index_stats()["total_chunks"]} chunks')